# ADI/OCO — MRC·PSM decorrect 단계별 작업
위에서 아래로 셀을 실행하면서 중간 테이블과 저장 파일을 확인합니다. SQL, 데이터 결합, 보정 함수가 이 노트북 안에 있으므로 여기서 직접 수정할 수 있습니다. 별도의 `decorrect/*.py` 파일은 필요하지 않습니다.

흐름: 설정 → LOT/wafer 선정 → RAWDATA → 최신 batch → 공정 정보 → 좌표/측정 테이블 → MRC → APC/PSM 이력 → PSM 평가 → raw → 최종 확인.

**이번 범위는 기적용 MRC·PSM을 되돌리는 부분까지입니다.** WK/RK 회귀, CPE 회귀, TROCS fitting, DiS, rework 판정은 포함하지 않습니다.

- 각 실행 셀에서 중간 결과를 저장하고 파일 링크를 표시합니다. 모든 행은 Parquet에 저장하며 큰 Excel은 `_sample_행수` 파일로 명확히 표시합니다.
- 원본 측정값은 그대로 보관합니다. ADI는 실제 PERSHOT, OCO는 계산한 PSM 값을 사용합니다.
- 설정·SQL·함수를 수정하면 해당 정의 셀과 그 결과에 의존하는 아래 셀을 다시 실행하세요. 정의 변경만으로 기존 결과 변수는 갱신되지 않습니다.
- live 모드는 사내 `bigdataquery`가 필요합니다. 저장된 조회 스냅샷은 replay 모드에서 DB 없이 읽을 수 있습니다.


## 00. 라이브러리와 실행 설정
먼저 아래 설정을 수정합니다. `DATA_MODE="replay"`이면 `REPLAY_DIR`에 기존 snapshots 폴더를 지정하세요.


In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import hashlib
import json
import logging
import os
import warnings
import numpy as np
import pandas as pd
from IPython.display import display, FileLink

pd.set_option("display.max_columns", 20)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


In [ ]:
@dataclass
class PipelineConfig:
    days: int = 8
    exclude_recent_days: int = 1
    expo_days: int = 10
    target_lotid: list[str] | str | None = None
    target_pstepseq: str | None = "VC075030"
    target_mstepseq: str | None = "VC075040"
    target_mstepseq_oco: str | None = "VC077251"
    db_user: str = "SIMAXP2"
    db_user_apc: str = "EAPCP2"
    slot_min: int = 9
    slot_max: int = 14
    rawdata_gap_seconds: int = 60
    apc_match_seconds: int = 60
    wafer_radius_um: float = 147000.0
    max_query_keys: int = 1000
    rawdata_row_limit: int = 10000000
    reference_date: str | None = None  # Python 측 ADI 날짜 필터용. SQL의 now()는 DB 기준.
    output_dir: str = "output_decorrect_notebook"
    save_parquet: bool = True
    save_excel: bool = False
    save_snapshots: bool = True
    max_excel_rows: int = 1048575

    def __post_init__(self):
        if self.days <= 0 or not 0 <= self.exclude_recent_days < self.days:
            raise ValueError("days > exclude_recent_days >= 0 이어야 합니다.")
        if self.slot_min > self.slot_max or self.max_query_keys <= 0 or self.rawdata_row_limit <= 0:
            raise ValueError("wafer 범위와 조회 제한을 확인하세요.")
        if self.rawdata_gap_seconds < 0 or self.apc_match_seconds < 0:
            raise ValueError("시간 간격은 0 이상이어야 합니다.")

DATA_MODE = "live"  # "live": 사내 DB 조회 / "replay": 저장된 조회 결과 사용
REPLAY_DIR = None  # 예: Path("output_decorrect_notebook/snapshots")
OUTPUT_DIR = Path.cwd() / "output_decorrect_notebook"
SAVE_PARQUET = True
SAVE_EXCEL = True
EXCEL_PREVIEW_ROWS = 10000  # 이보다 큰 테이블은 Excel 샘플 + 전체 Parquet을 저장
PREVIEW_ROWS = 10

cfg = PipelineConfig()
cfg.days = 8
cfg.exclude_recent_days = 1
cfg.expo_days = 10
cfg.target_lotid = None  # 예: ["B0K486.1", "B0K271.1"]
cfg.target_pstepseq = "VC075030"
cfg.target_mstepseq = "VC075040"
cfg.target_mstepseq_oco = "VC077251"
cfg.slot_min = 9
cfg.slot_max = 14
cfg.reference_date = None  # Python ADI 날짜 필터. SQL now()의 날짜를 바꾸지는 않음
cfg.output_dir = str(OUTPUT_DIR)
cfg.save_snapshots = True
cfg.save_parquet = SAVE_PARQUET
cfg.save_excel = SAVE_EXCEL
cfg.__post_init__()
if EXCEL_PREVIEW_ROWS < 1 or PREVIEW_ROWS < 1:
    raise ValueError("미리보기 행 수는 1 이상이어야 합니다.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("중간 결과 저장 폴더:", OUTPUT_DIR.resolve())


## 01. 공통 함수·조회 연결·중간 파일 저장
조회 조건이 같으면 MRC 결과를 재사용합니다. 같은 SQL로 DB를 다시 조회하려면 `source.release_cache()` 후 해당 조회 셀을 재실행하세요. 설정을 바꾸면 이 셀부터 다시 실행합니다.


In [ ]:
LOG = logging.getLogger("decorrect")

def require_columns(df, columns, label):
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"{label}: 필수 컬럼 누락 {missing}")

def numeric(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def integer(series):
    return pd.to_numeric(series, errors="coerce").astype("Int64")

def clean_columns(df):
    return df.loc[:, ~df.columns.duplicated()].copy() if df.columns.duplicated().any() else df

def lot_list(value):
    if value is None:
        return []
    if isinstance(value, str):
        value = value.strip().strip("()[]").split(",")
    return [str(v).strip().strip("'\"") for v in value if str(v).strip().strip("'\"")]

def quote(value):
    return "'" + str(value).replace("'", "''") + "'"

def in_filter(column, values, numeric_values=False):
    values = [v for v in values if pd.notna(v)]
    if not values:
        return "AND 1 = 0"  # 빈 IN 조건과 의도하지 않은 전체 조회를 막습니다.
    text = ",".join(str(int(v)) if numeric_values else quote(v) for v in values)
    return f"AND {column} IN ({text})"

def limited_keys(series, limit, label):
    values = series.dropna().unique().tolist()
    if len(values) > limit:
        warnings.warn(f"{label}: {len(values)} keys 중 원본과 동일하게 앞 {limit}개만 조회합니다.", stacklevel=2)
    return values[:limit]

def coord_key(series):
    rounded = pd.to_numeric(series, errors="coerce").round(2)
    return rounded.map(lambda value: f"{value:.2f}" if pd.notna(value) else "<NA>")

def psm_key(lot, chuck, x, y):
    return lot.astype(str).str.strip() + "_" + chuck.astype(str).str.strip() + "_" + coord_key(x) + "_" + coord_key(y)

def chuck_for_slot(series):
    slots = integer(series)
    return pd.Series(np.where(slots.fillna(0) % 2 == 1, "EX1", "EX2"), index=series.index).mask(slots.isna())

LOT_COLUMNS = (
    "l.lot_transn_seq,l.photo_transn_seq,l.impala_insert_time,l.db_user,l.`timestamp`,l.lotid,l.slotid,"
    "l.pstepseq,l.mstepseq,l.ppid,l.peqpid,l.chuckid,l.reticleid,l.steppitch_x,l.steppitch_y,l.mapshift_x,l.mapshift_y,"
    "l.dmargin_x,l.dmargin_y,l.outlr_resdl_spec_ratio,l.chip_x_qty,l.chip_y_qty,l.psm_mmo_ref_eqp_id,l.per_shot_mrc_id"
)

MRC_COLUMNS = [f"wk{i}" for i in range(1, 21)] + [f"rk{i}" for i in range(3, 21)]

PSM_COLUMNS = [f"mrc_k{i}_valn" for i in range(1, 73)]

def equals_filter(column, value):
    return f"AND {column} = {quote(value)}" if value else ""


In [ ]:
class QuerySource:
    def __init__(self, get_data=None, snapshot_dir=None, replay_dir=None):
        self.get_data = get_data
        self.snapshot_dir = Path(snapshot_dir) if snapshot_dir else None
        self.replay_dir = Path(replay_dir) if replay_dir else None
        self.cache = {}
        self.query_count = 0
        self.manifest = {"queries": {}}
        if self.replay_dir:
            self.manifest = json.loads((self.replay_dir / "manifest.json").read_text(encoding="utf-8"))

    def fetch(self, name, sql, reuse=True):
        signature = hashlib.sha256(" ".join(sql.split()).encode()).hexdigest()
        if self.replay_dir:
            saved = self.manifest["queries"].get(name)
            if not saved:
                raise FileNotFoundError(f"조회 스냅샷 없음: {name}. 해당 조건으로 live 실행을 먼저 하세요.")
            if saved["sql_sha256"] != signature:
                raise ValueError(f"{name}: 저장 당시 SQL 조건과 다릅니다. 같은 설정으로 replay하거나 새로 조회하세요.")
            frame = pd.read_parquet(self.replay_dir / f"{name}.parquet")
        elif reuse and signature in self.cache:
            frame = self.cache[signature]
            LOG.info("%s: 동일 SQL 조회 결과 재사용", name)
        else:
            if self.get_data is None:
                try:
                    import bigdataquery as bdq
                except ModuleNotFoundError as exc:
                    raise RuntimeError("live 실행에는 사내 bigdataquery가 필요합니다. 사내 환경에서 실행하거나 --replay로 저장된 스냅샷을 사용하세요.") from exc
                self.get_data = bdq.getData
            frame = self.get_data(sql)
            if not isinstance(frame, pd.DataFrame):
                raise TypeError(f"{name}: getData는 DataFrame을 반환해야 합니다.")
            self.query_count += 1
            if reuse:
                self.cache[signature] = frame
        LOG.info("%s: %s rows", name, len(frame))
        if self.snapshot_dir:
            self.snapshot_dir.mkdir(parents=True, exist_ok=True)
            frame.to_parquet(self.snapshot_dir / f"{name}.parquet", index=False)
            (self.snapshot_dir / f"{name}.sql").write_text(sql.strip() + "\n", encoding="utf-8")
            self.manifest["queries"][name] = {"rows": len(frame), "sql_sha256": signature}
        return frame.copy() if reuse else frame

    def save_manifest(self, cfg):
        if self.snapshot_dir:
            self.manifest["config"] = asdict(cfg)
            self.manifest["query_count"] = self.query_count
            self.manifest["created_at"] = pd.Timestamp.now().isoformat()
            (self.snapshot_dir / "manifest.json").write_text(json.dumps(self.manifest, ensure_ascii=False, indent=2), encoding="utf-8")

    def release_cache(self):
        self.cache.clear()

if DATA_MODE not in ("live", "replay"):
    raise ValueError("DATA_MODE는 live 또는 replay여야 합니다.")
if DATA_MODE == "replay" and REPLAY_DIR is None:
    raise ValueError("replay 모드에는 REPLAY_DIR을 지정하세요.")
snapshot_dir = OUTPUT_DIR / "snapshots" if DATA_MODE == "live" and cfg.save_snapshots else None
source = QuerySource(snapshot_dir=snapshot_dir, replay_dir=REPLAY_DIR if DATA_MODE == "replay" else None)
recorded_date = source.manifest.get("config", {}).get("reference_date")
cfg.reference_date = cfg.reference_date or recorded_date or pd.Timestamp.today().date().isoformat()

def fetch_frame(name, sql, reuse=True):
    frame = source.fetch(name, sql, reuse=reuse)
    source.save_manifest(cfg)  # 끝까지 돌리지 않아도 완료한 조회를 재실행할 수 있게 저장
    return frame

def snapshot_path(name):
    folder = source.replay_dir or source.snapshot_dir
    path = folder / f"{name}.parquet" if folder else None
    return path if path is not None and path.is_file() else None

STAGE_DIR = OUTPUT_DIR / "stages"
STAGE_DIR.mkdir(parents=True, exist_ok=True)
stage_files = {}

def save_stage(name, frame, columns=None, existing_parquet=None):
    print(f"[{name}] {len(frame):,} rows × {len(frame.columns)} columns")
    selected = [col for col in columns if col in frame] if columns else list(frame.columns)
    display(frame[selected].head(PREVIEW_ROWS))
    paths = []
    if SAVE_PARQUET:
        path = Path(existing_parquet) if existing_parquet is not None else STAGE_DIR / f"{name}.parquet"
        if existing_parquet is None:
            frame.to_parquet(path, index=False)
        paths.append(("parquet_full", path, len(frame)))
    if SAVE_EXCEL:
        limit = min(EXCEL_PREVIEW_ROWS, cfg.max_excel_rows)
        exported = frame.head(limit)
        suffix = f"_sample_{len(exported)}" if len(frame) > limit else ""
        path = STAGE_DIR / f"{name}{suffix}.xlsx"
        exported.to_excel(path, index=False, engine="openpyxl")
        paths.append(("excel_sample" if suffix else "excel_full", path, len(exported)))
    for kind, path, exported_rows in paths:
        key = f"{name}:{kind}"
        stage_files[key] = {"stage": name, "kind": kind, "total_rows": len(frame), "exported_rows": exported_rows, "path": str(path.resolve())}
        display(FileLink(os.path.relpath(path, Path.cwd()), result_html_prefix=f"{kind} ({exported_rows:,}행): "))
    pd.DataFrame(stage_files.values()).to_csv(OUTPUT_DIR / "saved_files.csv", index=False, encoding="utf-8-sig")

print("데이터 모드:", DATA_MODE, "| ADI 기준일:", cfg.reference_date)


## 02. OCO LOTINFO 조회
`sql_lotinfo_oco`와 `df_lotinfo_oco_query`를 확인합니다. 이 셀의 SQL 함수에서 OCO 조회 조건을 수정할 수 있습니다.


In [ ]:
def lotinfo_sql(cfg, mode, photo_ids=None):
    extra_days = 5 if mode == "OCO" else 10
    recent_days = max(0, cfg.exclude_recent_days - 3) if mode == "OCO" else cfg.exclude_recent_days
    metro = cfg.target_mstepseq_oco if mode == "OCO" else cfg.target_mstepseq
    selection = in_filter("l.lotid", lot_list(cfg.target_lotid)) if mode == "OCO" and cfg.target_lotid else ""
    if mode == "ADI":
        selection = in_filter("l.photo_transn_seq", photo_ids or [], True)
    row_limit = 1000 if mode == "OCO" else 100000
    return f"""
WITH lotinfo_raw AS (
  SELECT * FROM (
    SELECT {LOT_COLUMNS}, row_number() OVER (PARTITION BY l.lotid, l.slotid ORDER BY l.impala_insert_time DESC) AS rn
    FROM ees_ds_eai.apc_nautil_dat_lotinfo l
    WHERE l.impala_insert_time >= now() - interval {cfg.days + extra_days} days
      AND l.`timestamp` >= now() - interval {cfg.days + extra_days} days
      AND l.`timestamp` <= now() - interval {recent_days} days
      AND l.db_user = {quote(cfg.db_user)}
      {selection}
      {equals_filter('l.pstepseq', cfg.target_pstepseq)}
      {equals_filter('l.mstepseq', metro)}
  ) t WHERE t.rn = 1
)
SELECT * FROM lotinfo_raw LIMIT {row_limit}
"""

sql_lotinfo_oco = lotinfo_sql(cfg, "OCO")
df_lotinfo_oco_query = fetch_frame("lotinfo_oco", sql_lotinfo_oco)
save_stage("02_LOTINFO_OCO_QUERY", df_lotinfo_oco_query, existing_parquet=snapshot_path("lotinfo_oco"))


## 03. 동일 노광 이력의 ADI LOTINFO 조회
OCO의 photo_transn_seq를 이용합니다. 원본처럼 최대 조회 key 수를 유지하며 초과 시 알려줍니다.


In [ ]:
photo_ids = limited_keys(df_lotinfo_oco_query["photo_transn_seq"], cfg.max_query_keys, "ADI LOTINFO") if not df_lotinfo_oco_query.empty else []
sql_lotinfo_adi = lotinfo_sql(cfg, "ADI", photo_ids)
df_lotinfo_adi_query = fetch_frame("lotinfo_adi", sql_lotinfo_adi) if photo_ids else pd.DataFrame()
save_stage("03_LOTINFO_ADI_QUERY", df_lotinfo_adi_query, existing_parquet=snapshot_path("lotinfo_adi"))


## 04. 날짜·동일 wafer·slot 범위 적용
`df_lotinfo_adi`, `df_lotinfo_oco`가 이후의 분석 대상입니다. 대상 선정 로직은 이 셀에서 수정합니다.


In [ ]:
def select_lot_pair(adi, oco, cfg):
    if adi.empty or oco.empty:
        return adi.iloc[:0].copy(), oco.iloc[:0].copy(), pd.DataFrame()
    adi, oco = adi.copy(), oco.copy()
    for frame in (adi, oco):
        require_columns(frame, ["photo_transn_seq", "slotid", "lot_transn_seq", "lotid", "timestamp"], "LOTINFO")
        frame["photo_transn_seq"], frame["slotid"] = integer(frame["photo_transn_seq"]), integer(frame["slotid"])
    adi["timestamp"] = pd.to_datetime(adi["timestamp"], errors="coerce")
    today = pd.Timestamp(cfg.reference_date).normalize() if cfg.reference_date else pd.Timestamp.today().normalize()
    start, end = today - pd.Timedelta(days=cfg.days), today - pd.Timedelta(days=cfg.exclude_recent_days)
    adi = adi[adi["timestamp"].ge(start) & adi["timestamp"].lt(end)].copy()
    keys = ["photo_transn_seq", "slotid"]
    common = oco[keys].dropna().drop_duplicates().merge(adi[keys].dropna().drop_duplicates(), on=keys, how="inner")
    adi, oco = [frame.merge(common, on=keys, how="inner").sort_values(keys).reset_index(drop=True) for frame in (adi, oco)]
    adi, oco = [frame[frame["slotid"].between(cfg.slot_min, cfg.slot_max)].copy() for frame in (adi, oco)]
    for frame in (adi, oco):
        frame["slotid"] = frame["slotid"].astype(int)
    oco_extra = oco[keys + ["lot_transn_seq", "timestamp", "mstepseq"]].drop_duplicates(keys)
    oco_extra = oco_extra.rename(columns={"lot_transn_seq": "lot_transn_seq_oco", "timestamp": "timestamp_oco", "mstepseq": "mstepseq_oco"})
    saved = adi.merge(oco_extra, on=keys, how="left")
    if "per_shot_mrc_id" in saved:
        parts = saved["per_shot_mrc_id"].astype(str).str.split("/").str[-1].str.split("_")
        saved["base_eqpid"] = parts.str[0]
        saved["psm_update_time"] = parts.str[1].astype("string").str.replace(r"^20", "", regex=True).str[:10]
    return adi, oco, saved

df_lotinfo_adi, df_lotinfo_oco, df_lotinfo_save = select_lot_pair(df_lotinfo_adi_query, df_lotinfo_oco_query, cfg)
save_stage("04_LOTINFO_ADI_SELECTED", df_lotinfo_adi)
save_stage("04_LOTINFO_OCO_SELECTED", df_lotinfo_oco)
save_stage("04_LOTINFO_PAIRED", df_lotinfo_save)


## 05. RAWDATA / TESTDATA / PERSHOT 조회
가공 전 조회 결과를 확인합니다. 큰 조회 결과는 snapshots의 전체 Parquet을 연결해서 중복 저장을 줄입니다.


In [ ]:
def rawdata_sql(cfg, lots):
    ids = limited_keys(lots["lot_transn_seq"], cfg.max_query_keys, "RAWDATA")
    return f"""
SELECT r.lot_transn_seq,r.slot_id,r.transn_occur_date,r.data_kind,r.test_point_no,r.coordinate_x,r.coordinate_y,r.value_x,r.value_y,r.mrc_x_valn,r.mrc_y_valn
FROM ees_ds_eai.apc_nautil_dat_rawdata r
WHERE r.impala_insert_time >= now() - interval {cfg.days + 10} days AND r.db_user = {quote(cfg.db_user)}
  {in_filter('r.lot_transn_seq', ids, True)} AND r.data_kind IN ('RAWDATA','TESTDATA','PERSHOT')
LIMIT {cfg.rawdata_row_limit}
"""

sql_rawdata_oco = rawdata_sql(cfg, df_lotinfo_oco) if not df_lotinfo_oco.empty else None
sql_rawdata_adi = rawdata_sql(cfg, df_lotinfo_adi) if not df_lotinfo_adi.empty else None
df_rawdata_oco_query = fetch_frame("rawdata_oco", sql_rawdata_oco, reuse=False) if sql_rawdata_oco else pd.DataFrame()
df_rawdata_adi_query = fetch_frame("rawdata_adi", sql_rawdata_adi, reuse=False) if sql_rawdata_adi else pd.DataFrame()
save_stage("05_RAWDATA_OCO_QUERY", df_rawdata_oco_query, existing_parquet=snapshot_path("rawdata_oco"))
save_stage("05_RAWDATA_ADI_QUERY", df_rawdata_adi_query, existing_parquet=snapshot_path("rawdata_adi"))


## 06. 종류별 최신 batch 선택
RAWDATA / TESTDATA / PERSHOT 각각 마지막 batch를 남깁니다. 원본 조회와 선택 후의 행 수를 비교하세요.


In [ ]:
def latest_raw_batch(raw, lots, gap_seconds=60):
    if raw.empty or lots.empty:
        return raw.iloc[:0].copy()
    require_columns(raw, ["lot_transn_seq", "slot_id", "data_kind", "transn_occur_date"], "RAWDATA")
    df = raw.copy()
    df["transn_occur_date"] = pd.to_datetime(df["transn_occur_date"].astype(str), errors="coerce")
    df["slot_id"] = integer(df["slot_id"])
    df = df.dropna(subset=["slot_id", "transn_occur_date"]).copy()
    df["slot_id"] = df["slot_id"].astype(int)
    keys = lots[["lot_transn_seq", "slotid"]].drop_duplicates().copy()
    keys["lot_transn_seq"], keys["slotid"] = integer(keys["lot_transn_seq"]), integer(keys["slotid"])
    df["lot_transn_seq"] = integer(df["lot_transn_seq"])
    df = df.merge(keys.dropna(), left_on=["lot_transn_seq", "slot_id"], right_on=["lot_transn_seq", "slotid"], how="inner")
    groups = ["lot_transn_seq", "slot_id", "data_kind"]
    df = df.sort_values(groups + ["transn_occur_date"]).reset_index(drop=True)
    previous = df.groupby(groups)["transn_occur_date"].shift()
    starts = previous.isna() | (df["transn_occur_date"] - previous).dt.total_seconds().gt(gap_seconds)
    df["_batch"] = starts.astype(int).groupby([df[col] for col in groups]).cumsum()
    # transform으로 최대 batch를 붙이면 원본 merge와 같은 행 선택을 중간 테이블 없이 수행합니다.
    last = df.groupby(groups)["_batch"].transform("max")
    return df[df["_batch"].eq(last)].drop(columns="_batch").reset_index(drop=True)

df_rawdata_oco = latest_raw_batch(df_rawdata_oco_query, df_lotinfo_oco, cfg.rawdata_gap_seconds)
df_rawdata_adi = latest_raw_batch(df_rawdata_adi_query, df_lotinfo_adi, cfg.rawdata_gap_seconds)
save_stage("06_RAWDATA_OCO_LATEST", df_rawdata_oco)
save_stage("06_RAWDATA_ADI_LATEST", df_rawdata_adi)
source.release_cache()


## 07. EXPO·F1 공정 정보 조회
노광 이력의 보정 index와 장비 정보를 가져옵니다. `df_paramdata`는 최신 F1 선택 전의 조회 결과입니다.


In [ ]:
def expo_sql(cfg, lots):
    ids = limited_keys(lots["lotid"], cfg.max_query_keys, "EXPO")
    return f"""
SELECT CAST(e.apc_hist_index_no AS BIGINT) AS apc_hist_index_no,e.apc_trocs_hist_index_no,e.impala_insert_time,e.seq_no,e.lot_id,e.slot_id,e.photo_step_seq,e.metro_step_seq,e.photo_date,e.photo_transn_seq,e.mmo_mrc_ref_eqp_id
FROM ees_ds_eai.apc_expo_overlay_lot e
WHERE e.impala_insert_time >= now() - interval {cfg.expo_days + 5} days AND e.db_user = {quote(cfg.db_user)}
  {in_filter('e.lot_id', ids)} {equals_filter('e.photo_step_seq', cfg.target_pstepseq)}
LIMIT 100000
"""

def f1_sql(cfg, lots):
    ids = limited_keys(lots["photo_transn_seq"], cfg.max_query_keys, "F1")
    return f"""
SELECT p.photo_transn_seq,p.lotid,p.slotid,p.base_eqp_id1,p.impala_insert_time
FROM ees_ds_eai.apc_nautil_paramdata p
WHERE p.impala_insert_time >= now() - interval {cfg.days + 5} days AND p.db_user = {quote(cfg.db_user)}
  {in_filter('p.photo_transn_seq', ids, True)} AND p.subname = 'F1' {equals_filter('p.mstepseq', cfg.target_mstepseq)}
LIMIT 1000000
"""

def prepare_expo(expo, lots):
    if expo.empty or lots.empty:
        return expo.iloc[:0].copy()
    expo = expo.sort_values("photo_date").drop_duplicates(["lot_id", "slot_id"], keep="last").copy()
    expo["slot_id"] = integer(expo["slot_id"])
    return expo.merge(lots[["lotid", "slotid"]].drop_duplicates(), left_on=["lot_id", "slot_id"], right_on=["lotid", "slotid"], how="inner").sort_values(["photo_transn_seq", "slotid"]).reset_index(drop=True)

sql_expo = expo_sql(cfg, df_lotinfo_adi) if not df_lotinfo_adi.empty else None
sql_f1 = f1_sql(cfg, df_lotinfo_adi) if not df_lotinfo_adi.empty else None
df_expo_overlay_query = fetch_frame("expo", sql_expo) if sql_expo else pd.DataFrame()
df_paramdata = fetch_frame("f1", sql_f1) if sql_f1 else pd.DataFrame()
df_expo_overlay = prepare_expo(df_expo_overlay_query, df_lotinfo_adi)
save_stage("07_EXPO_SELECTED", df_expo_overlay)
save_stage("07_F1_QUERY", df_paramdata, existing_parquet=snapshot_path("f1"))


## 08. 측정값·좌표·메타데이터 결합
그룹, 좌표, ID, edge 여부를 계산합니다. `df_adi_measured`, `df_oco_measured`는 MRC decorrect 전 단계이며 원래 X_reg/Y_reg가 남아 있습니다.


In [ ]:
RENAME = {
    "pstepseq": "STEPSEQ", "lotid": "LOT_ID", "slotid": "Wafer", "peqpid": "P_EQPID", "ppid": "Photo_PPID",
    "chuckid": "ChuckID", "reticleid": "ReticleID", "steppitch_x": "STEP_PITCH_X", "steppitch_y": "STEP_PITCH_Y",
    "mapshift_x": "MAP_SHIFT_X", "mapshift_y": "MAP_SHIFT_Y", "dmargin_x": "Dmargin_X", "dmargin_y": "Dmargin_Y",
    "outlr_resdl_spec_ratio": "Outlier_Spec_Ratio", "chip_x_qty": "CHIP_X_NUM", "chip_y_qty": "CHIP_Y_NUM",
    "mmo_mrc_ref_eqp_id": "MMO_MRC_EQP", "mstepseq": "M_STEPSEQ", "timestamp": "M_TIME",
}

FINAL_COLUMNS = [
    "UNIQUE_ID", "UNIQUE_ID2", "UNIQUE_ID3", "UNIQUE_ID4", "STEPSEQ", "LOT_ID", "Wafer",
    "P_EQPID", "Photo_PPID", "P_TIME", "M_TIME", "ChuckID", "ReticleID", "Base_EQP1", "MMO_MRC_EQP", "GROUP", "TEST", "DieX", "DieY",
    "STEP_PITCH_X", "STEP_PITCH_Y", "MAP_SHIFT_X", "MAP_SHIFT_Y", "coordinate_X", "coordinate_Y",
    "fcp_x", "fcp_y", "wf_x", "wf_y", "radius", "CHIP_X_NUM", "CHIP_Y_NUM", "Outlier_Spec_Ratio", "Dmargin_X", "Dmargin_Y",
    "X_reg", "Y_reg", "MRC_RX", "MRC_RY", "PSM_X", "PSM_Y", "Is_Edge_Shot", "Max_Corner_Distance",
    "M_STEPSEQ", "photo_transn_seq", "apc_hist_index_no", "apc_trocs_hist_index_no",
]

def assemble_measurements(raw, lots, expo, f1, mode, cfg):
    if raw.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)
    df = raw[raw["data_kind"].eq("RAWDATA")].rename(columns={"test_point_no": "TEST", "coordinate_x": "DieX", "coordinate_y": "DieY", "value_x": "X_reg", "value_y": "Y_reg"}).copy()
    if df.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)
    df = clean_columns(df).merge(lots, left_on=["lot_transn_seq", "slot_id"], right_on=["lot_transn_seq", "slotid"], how="left", validate="m:1", suffixes=("", "_lotinfo"))
    test = raw[raw["data_kind"].eq("TESTDATA")].rename(columns={"test_point_no": "TEST", "value_x": "coordinate_X", "value_y": "coordinate_Y", "mrc_x_valn": "MRC_RX", "mrc_y_valn": "MRC_RY"})
    if not test.empty:
        columns = ["lot_transn_seq", "slot_id", "TEST", "coordinate_X", "coordinate_Y", "MRC_RX", "MRC_RY"]
        df = df.merge(test[columns], on=columns[:3], how="left", validate="m:1", suffixes=("", "_test"))
    for col in ["coordinate_X", "coordinate_Y", "MRC_RX", "MRC_RY"]:
        if col not in df:
            df[col] = np.nan
    if mode == "ADI":
        pershot = raw[raw["data_kind"].eq("PERSHOT")].rename(columns={"test_point_no": "TEST", "coordinate_x": "DieX", "coordinate_y": "DieY", "value_x": "PSM_X", "value_y": "PSM_Y"})
        columns = ["lot_transn_seq", "slot_id", "TEST", "DieX", "DieY", "PSM_X", "PSM_Y"]
        if not pershot.empty:
            df = df.merge(pershot[columns], on=columns[:5], how="left", validate="m:1", suffixes=("", "_psm"))
        for col in ["PSM_X", "PSM_Y"]:
            if col not in df:
                df[col] = np.nan
    if not expo.empty:
        columns = ["lotid", "slot_id", "photo_transn_seq", "photo_date", "apc_hist_index_no", "apc_trocs_hist_index_no", "mmo_mrc_ref_eqp_id"]
        df = df.merge(expo[columns].rename(columns={"photo_date": "P_TIME", "photo_transn_seq": "photo_transn_seq_expo"}), on=["lotid", "slot_id"], how="left", validate="m:1")
    else:
        for col in ["P_TIME", "apc_hist_index_no", "apc_trocs_hist_index_no", "mmo_mrc_ref_eqp_id"]:
            df[col] = pd.NA
    # 원본의 photo_transn_seq_x/y 충돌 때문에 빠지던 F1 메타데이터만 명시적으로 연결합니다.
    if not f1.empty:
        f1 = f1.copy()
        f1["photo_transn_seq"], f1["slotid"] = integer(f1["photo_transn_seq"]), integer(f1["slotid"])
        f1["_f1_lot_key"] = f1["lotid"].astype(str).str.strip()
        df["_f1_lot_key"] = df["lotid"].astype(str).str.strip()
        keys = ["photo_transn_seq", "_f1_lot_key", "slotid"]
        f1 = f1.sort_values("impala_insert_time").drop_duplicates(keys, keep="last") if "impala_insert_time" in f1 else f1.drop_duplicates(keys, keep="last")
        df = df.merge(f1[keys + ["base_eqp_id1"]], on=keys, how="left", validate="m:1").rename(columns={"base_eqp_id1": "Base_EQP1"})
        df = df.drop(columns="_f1_lot_key")
    df = clean_columns(df.rename(columns=RENAME)).reset_index(drop=True)
    numeric(df, ["TEST", "X_reg", "Y_reg", "PSM_X", "PSM_Y", "MRC_RX", "MRC_RY"])
    df["GROUP"] = pd.cut(df["TEST"], bins=[0, 80, 160, 240], labels=["E1", "E2", "E3"])
    if mode == "OCO":
        group = "E2" if (cfg.target_mstepseq_oco or "")[2:5] == "077" else "E1"
        df = df[df["GROUP"].eq(group)].copy()
    return add_geometry_and_ids(df, cfg)

def add_geometry_and_ids(df, cfg):
    if df.empty:
        return pd.DataFrame(columns=FINAL_COLUMNS)
    df = df.reset_index(drop=True)
    columns = ["DieX", "DieY", "STEP_PITCH_X", "STEP_PITCH_Y", "MAP_SHIFT_X", "MAP_SHIFT_Y", "coordinate_X", "coordinate_Y"]
    require_columns(df, columns, "좌표 계산")
    for col in columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors="coerce")
    df["fcp_x"] = df["DieX"] * df["STEP_PITCH_X"] + df["MAP_SHIFT_X"]
    df["fcp_y"] = df["DieY"] * df["STEP_PITCH_Y"] + df["MAP_SHIFT_Y"]
    df["wf_x"], df["wf_y"] = df["fcp_x"] + df["coordinate_X"], df["fcp_y"] + df["coordinate_Y"]
    df["radius"] = np.sqrt(df["wf_x"] ** 2 + df["wf_y"] ** 2)
    fields = ["STEPSEQ", "P_EQPID", "Photo_PPID", "MMO_MRC_EQP", "P_TIME", "M_TIME", "LOT_ID", "Wafer"]
    text = {col: df[col].astype("string").fillna("") for col in fields + ["GROUP", "TEST", "DieX", "DieY"]}
    base = text[fields[0]]
    for col in fields[1:]:
        base = base + "_" + text[col]
    df["UNIQUE_ID3"], df["UNIQUE_ID"] = base, base + "_" + text["GROUP"]
    df["UNIQUE_ID2"] = base + "_" + text["TEST"] + "_" + text["DieX"] + "_" + text["DieY"] + "_" + text["GROUP"]
    df["UNIQUE_ID4"] = base + "_" + text["DieX"] + "_" + text["DieY"] + "_" + text["GROUP"]
    x, y = df["fcp_x"].to_numpy(float), df["fcp_y"].to_numpy(float)
    sx, sy = df["STEP_PITCH_X"].to_numpy(float) / 2, df["STEP_PITCH_Y"].to_numpy(float) / 2
    distances = [(x + dx * sx) ** 2 + (y + dy * sy) ** 2 for dx, dy in [(-1, -1), (-1, 1), (1, -1), (1, 1)]]
    df["Max_Corner_Distance"] = np.sqrt(np.maximum.reduce(distances))
    df["Is_Edge_Shot"] = df["Max_Corner_Distance"].gt(cfg.wafer_radius_um)
    return df[[col for col in FINAL_COLUMNS if col in df]].copy()

df_adi_measured = assemble_measurements(df_rawdata_adi, df_lotinfo_adi, df_expo_overlay, df_paramdata, "ADI", cfg)
df_oco_measured = assemble_measurements(df_rawdata_oco, df_lotinfo_oco, df_expo_overlay, df_paramdata, "OCO", cfg)
save_stage("08_ADI_MEASURED", df_adi_measured)
save_stage("08_OCO_MEASURED", df_oco_measured)


## 09. 기적용 MRC(CMP) 계수 조회·최신 이력 선택
원본처럼 ADI와 OCO 모두 ADI mstepseq의 CMP 계수를 사용합니다. 현재 mrc_sql의 조건은 여기서 수정합니다.


In [ ]:
def mrc_sql(cfg, frame):
    # 원본 007.py와 같이 ADI/OCO 모두 ADI mstepseq의 CMP 계수를 사용합니다.
    ids = limited_keys(frame["photo_transn_seq"], cfg.max_query_keys, "CMP")
    columns = ",".join(f"p.{col}" for col in MRC_COLUMNS)
    return f"""
SELECT p.photo_transn_seq,p.lotid,p.slotid,p.subname,{columns},p.impala_insert_time
FROM ees_ds_eai.apc_nautil_paramdata p
WHERE p.impala_insert_time >= now() - interval {cfg.days + 10} days AND p.db_user = {quote(cfg.db_user)}
  {in_filter('p.photo_transn_seq', ids, True)} AND p.subname = 'CMP' {equals_filter('p.mstepseq', cfg.target_mstepseq)}
LIMIT 100000
"""

def prepare_mrc_input(mrc):
    if mrc.empty:
        return pd.DataFrame(columns=["photo_transn_seq", "lotid", "slotid"] + MRC_COLUMNS)
    require_columns(mrc, ["photo_transn_seq", "lotid", "slotid", "impala_insert_time"] + MRC_COLUMNS, "CMP")
    df = mrc.copy()
    df = df.sort_values("impala_insert_time").drop_duplicates(["photo_transn_seq", "lotid", "slotid"], keep="last")
    df["photo_transn_seq"], df["slotid"] = integer(df["photo_transn_seq"]), integer(df["slotid"])
    return numeric(df, MRC_COLUMNS)

sql_kmrc_adi = mrc_sql(cfg, df_adi_measured) if not df_adi_measured.empty else None
sql_kmrc_oco = mrc_sql(cfg, df_oco_measured) if not df_oco_measured.empty else None
df_kmrc_adi_query = fetch_frame("mrc_adi", sql_kmrc_adi) if sql_kmrc_adi else pd.DataFrame()
df_kmrc_oco_query = fetch_frame("mrc_oco", sql_kmrc_oco) if sql_kmrc_oco else pd.DataFrame()
df_kmrc_adi = prepare_mrc_input(df_kmrc_adi_query)
df_kmrc_oco = prepare_mrc_input(df_kmrc_oco_query)
save_stage("09_MRC_ADI_INPUT", df_kmrc_adi)
save_stage("09_MRC_OCO_INPUT", df_kmrc_oco)


## 10. MRC decorrect
기적용 MRC를 원본의 항 순서·스케일·부호로 계산합니다. X_reg_demrc/Y_reg_demrc와 mrc_status를 확인하세요. 새로운 회귀를 하는 단계는 아닙니다.


In [ ]:
def osr_wk20p_rk20p(x, y, rx, ry):
    X_dx = np.vstack([
        np.ones(len(x)),
        x / 1e6, -y / 1e6,
        (x ** 2) / 1e12, (x * y) / 1e12, (y ** 2) / 1e12,
        (x ** 3) / 1e15, (x ** 2 * y) / 1e15, (x * y ** 2) / 1e15, (y ** 3) / 1e15,
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12
    ]).T

    X_dy = np.vstack([
        np.ones(len(y)),
        y / 1e6, x / 1e6,
        (y ** 2) / 1e12, (y * x) / 1e12, (x ** 2) / 1e12,
        (y ** 3) / 1e15, (y ** 2 * x) / 1e15, (y * x ** 2) / 1e15, (x ** 3) / 1e15,
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12, (rx ** 3) / 1e12
    ]).T

    return X_dx, X_dy

MRC_X_COLUMNS = [f"wk{i}" for i in range(1, 21, 2)] + [f"rk{i}" for i in range(3, 21, 2)]

MRC_Y_COLUMNS = [f"wk{i}" for i in range(2, 21, 2)] + [f"rk{i}" for i in range(4, 21, 2)]

def apply_mrc(frame, mrc):
    df = frame.copy()
    for col in ["mrc_fit_x", "mrc_fit_y", "X_reg_demrc", "Y_reg_demrc"]:
        df[col] = np.nan
    df["mrc_status"] = "missing_mrc"
    if df.empty:
        return df
    require_columns(df, ["photo_transn_seq", "LOT_ID", "Wafer", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y", "X_reg", "Y_reg"], "MRC 적용")
    numeric(df, ["X_reg", "Y_reg", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y"])
    if not mrc.empty:
        require_columns(mrc, ["photo_transn_seq", "lotid", "slotid"] + MRC_COLUMNS, "MRC 계수")
        # 계수를 모든 측정행에 merge하면 큰 복제 테이블이 생기므로 wafer별 lookup만 만듭니다.
        keys = list(zip(integer(mrc["photo_transn_seq"]), mrc["lotid"], integer(mrc["slotid"])))
        bx = np.ascontiguousarray(mrc[MRC_X_COLUMNS].to_numpy(float))
        by = np.ascontiguousarray(mrc[MRC_Y_COLUMNS].to_numpy(float))
        lookup = {key: i for i, key in reversed(list(enumerate(keys))) if all(pd.notna(v) for v in key)}
        group_keys = pd.DataFrame({"photo": integer(df["photo_transn_seq"]), "lot": df["LOT_ID"], "slot": integer(df["Wafer"])}, index=df.index)
        coordinates = [df[col].to_numpy(float) for col in ["fcp_x", "fcp_y", "coordinate_X", "coordinate_Y"]]
        fit_x, fit_y, status = np.full(len(df), np.nan), np.full(len(df), np.nan), df["mrc_status"].to_numpy(object)
        for key, positions in group_keys.groupby(["photo", "lot", "slot"], sort=False).indices.items():
            idx = lookup.get(key)
            if idx is None:
                continue
            if np.isnan(bx[idx]).any() or np.isnan(by[idx]).any():
                status[positions] = "invalid_mrc_k"
                continue
            mx, my = osr_wk20p_rk20p(*(values[positions] for values in coordinates))
            fit_x[positions], fit_y[positions] = mx.dot(bx[idx]) * -1, my.dot(by[idx]) * -1
            status[positions] = "matched"
        df["mrc_fit_x"], df["mrc_fit_y"], df["mrc_status"] = fit_x, fit_y, status
    df["X_reg_demrc"], df["Y_reg_demrc"] = df["X_reg"] - df["mrc_fit_x"], df["Y_reg"] - df["mrc_fit_y"]
    return df

df_adi_demrc = apply_mrc(df_adi_measured, df_kmrc_adi)
df_oco_demrc = apply_mrc(df_oco_measured, df_kmrc_oco)
mrc_preview = ["LOT_ID", "Wafer", "TEST", "X_reg", "Y_reg", "mrc_fit_x", "mrc_fit_y", "X_reg_demrc", "Y_reg_demrc", "mrc_status"]
save_stage("10_ADI_DEMRC", df_adi_demrc, columns=mrc_preview)
save_stage("10_OCO_DEMRC", df_oco_demrc, columns=mrc_preview)


## 11. APC 일반 이력·EEXY 연결
EEXY와 apc_trocs_hist_index_no는 원본의 PSM 계수 조회에 필요한 연결 정보입니다. TROCS 계수 평가를 수행하지는 않습니다.


In [ ]:
def history_sql(cfg, lots, eexy=False):
    ids = limited_keys(lots["lotid"], cfg.max_query_keys, "APC LOT HIST")
    columns = "CAST(apc_hist_index_no AS BIGINT) AS apc_trocs_hist_index_no,lot_id,step_seq,eqp_id,sub_eqp_id,tkin_tsdt,event_tsdt AS event_tsdt_eexy"
    if not eexy:
        columns = "CAST(apc_hist_index_no AS BIGINT) AS apc_hist_index_no,apc_group_name,lot_id,step_seq,eqp_id,sub_eqp_id,ppid,reticle_id,run_type,event_tsdt,tkin_tsdt"
    condition = "=" if eexy else "!="
    return f"""
SELECT {columns} FROM ees_ds_eai.apc_apc_lot_hist
WHERE db_user = {quote(cfg.db_user_apc)} AND event_tsdt >= now() - interval {cfg.days + 5} days
  AND event_tsdt <= now() - interval {cfg.exclude_recent_days} days AND apc_group_name {condition} 'PHOTO_EEXY_GRP' AND save_type != 'A'
  {in_filter('lot_id', ids)} {equals_filter('step_seq', cfg.target_pstepseq)}
"""

HISTORY_KEYS = ["lot_id", "step_seq", "eqp_id", "sub_eqp_id"]

HISTORY_COLUMNS = ["apc_hist_index_no", "apc_trocs_hist_index_no", "lot_id", "step_seq", "eqp_id", "sub_eqp_id", "ppid", "reticle_id", "run_type", "event_tsdt", "tkin_tsdt"]

PSM_META = ["event_tsdt", "tkin_tsdt", "apc_hist_index_no", "lot_id", "step_seq", "eqp_id", "sub_eqp_id", "ppid", "reticle_id", "run_type", "pos_x_valn", "pos_y_valn"]

def match_apc_history(main, eexy, tolerance_seconds=60):
    if main.empty:
        return pd.DataFrame(columns=HISTORY_COLUMNS)
    main, eexy = main.copy(), eexy.copy()
    if eexy.empty:
        out = main
        out["apc_trocs_hist_index_no"] = pd.NA
    else:
        for frame in (main, eexy):
            for col in HISTORY_KEYS:
                frame[col] = frame[col].astype(str).str.strip()
            frame["tkin_tsdt"] = pd.to_datetime(frame["tkin_tsdt"].astype(str), errors="coerce")
        main["event_tsdt"] = pd.to_datetime(main["event_tsdt"].astype(str), errors="coerce")
        eexy["event_tsdt_eexy"] = pd.to_datetime(eexy["event_tsdt_eexy"].astype(str), errors="coerce")
        main["apc_hist_index_no"], eexy["apc_trocs_hist_index_no"] = integer(main["apc_hist_index_no"]), integer(eexy["apc_trocs_hist_index_no"])
        left = main.dropna(subset=["tkin_tsdt"]).sort_values("tkin_tsdt")
        right = eexy.dropna(subset=["tkin_tsdt"]).sort_values("event_tsdt_eexy").drop_duplicates(HISTORY_KEYS + ["tkin_tsdt"], keep="last").sort_values("tkin_tsdt")
        out = pd.merge_asof(left, right[HISTORY_KEYS + ["tkin_tsdt", "apc_trocs_hist_index_no"]], on="tkin_tsdt", by=HISTORY_KEYS, tolerance=pd.Timedelta(seconds=tolerance_seconds), direction="nearest")
        invalid = main[main["tkin_tsdt"].isna()].copy()
        if not invalid.empty:
            invalid["apc_trocs_hist_index_no"] = pd.NA
            out = pd.concat([out, invalid], ignore_index=True)
    out["event_tsdt"] = pd.to_datetime(out["event_tsdt"], errors="coerce")
    out["apc_hist_index_no"], out["apc_trocs_hist_index_no"] = integer(out["apc_hist_index_no"]), integer(out["apc_trocs_hist_index_no"])
    # rich process key로 바꾸지 않습니다. 원본은 LOT/chuck의 마지막 이력을 선택합니다.
    return out.sort_values("event_tsdt").drop_duplicates(["lot_id", "sub_eqp_id"], keep="last").reset_index(drop=True)

sql_lot_hist_main = history_sql(cfg, df_lotinfo_adi) if not df_lotinfo_adi.empty else None
sql_lot_hist_eexy = history_sql(cfg, df_lotinfo_adi, eexy=True) if not df_lotinfo_adi.empty else None
df_lot_hist_main = fetch_frame("apc_main", sql_lot_hist_main) if sql_lot_hist_main else pd.DataFrame()
df_lot_hist_eexy = fetch_frame("apc_eexy", sql_lot_hist_eexy) if sql_lot_hist_eexy else pd.DataFrame()
df_lot_hist_apc = match_apc_history(df_lot_hist_main, df_lot_hist_eexy, cfg.apc_match_seconds)
save_stage("11_APC_MAIN_QUERY", df_lot_hist_main, existing_parquet=snapshot_path("apc_main"))
save_stage("11_APC_EEXY_QUERY", df_lot_hist_eexy, existing_parquet=snapshot_path("apc_eexy"))
save_stage("11_APC_MATCHED", df_lot_hist_apc)


## 12. 기적용 PSM 계수·shot 매칭 테이블
조회 결과와 LOT/chuck/shot 좌표가 연결된 df_psm_input을 각각 확인합니다. 조회한 shot 중심 좌표는 mm에서 µm로 변환합니다.


In [ ]:
def shot_sql(cfg, history):
    ids = history["apc_trocs_hist_index_no"].dropna().unique().tolist()
    columns = ",".join(PSM_COLUMNS)  # 이번 범위에는 TROCS rk1~72 조회가 필요하지 않습니다.
    return f"""
SELECT CAST(apc_hist_index_no AS BIGINT) AS apc_hist_index_no,pos_x_valn,pos_y_valn,{columns}
FROM ees_ds_eai.apc_apc_photo_shot_input_hist
WHERE db_user = {quote(cfg.db_user_apc)} AND impala_insert_time >= now() - interval {cfg.days + 10} days
  {in_filter('CAST(apc_hist_index_no AS BIGINT)', ids, True)}
"""

def prepare_psm_input(shot, history, lots):
    if shot.empty or history.empty:
        return pd.DataFrame(columns=PSM_META + PSM_COLUMNS + ["match_key"])
    require_columns(shot, ["apc_hist_index_no", "pos_x_valn", "pos_y_valn"] + PSM_COLUMNS, "PSM INPUT")
    shot, history = shot.copy(), history.copy()
    numeric(shot, ["pos_x_valn", "pos_y_valn"] + PSM_COLUMNS)
    shot["apc_hist_index_no"] = integer(shot["apc_hist_index_no"])
    for col in ["apc_hist_index_no", "apc_trocs_hist_index_no"]:
        history[col] = integer(history[col])
    for col in ["event_tsdt", "tkin_tsdt"]:
        history[col] = pd.to_datetime(history[col], errors="coerce")
    subset = history[HISTORY_COLUMNS].dropna(subset=["apc_trocs_hist_index_no"]).sort_values(["event_tsdt", "tkin_tsdt"]).drop_duplicates("apc_trocs_hist_index_no", keep="last")
    if subset.empty:
        return pd.DataFrame(columns=PSM_META + PSM_COLUMNS + ["match_key"])
    joined = shot.merge(subset, left_on="apc_hist_index_no", right_on="apc_trocs_hist_index_no", how="left", validate="m:1", suffixes=("_shot", "_lot"))
    joined = joined.rename(columns={"apc_hist_index_no_lot": "apc_hist_index_no"})
    df = joined[PSM_META + PSM_COLUMNS].copy()
    df["pos_x_valn"], df["pos_y_valn"] = df["pos_x_valn"] * 1000, df["pos_y_valn"] * 1000
    df["lot_id"], df["sub_eqp_id"] = df["lot_id"].astype(str).str.strip(), df["sub_eqp_id"].astype(str).str.strip()
    keys = pd.DataFrame({"lot_id": lots["lotid"].astype(str), "sub_eqp_id": chuck_for_slot(lots["slotid"])})
    df = df.merge(keys.dropna().drop_duplicates(), on=["lot_id", "sub_eqp_id"], how="inner")
    df["fcp_x_round"], df["fcp_y_round"] = df["pos_x_valn"].round(2), df["pos_y_valn"].round(2)
    df["match_key"] = psm_key(df["lot_id"], df["sub_eqp_id"], df["fcp_x_round"], df["fcp_y_round"])
    return df

has_shot_ids = not df_lot_hist_apc.empty and df_lot_hist_apc["apc_trocs_hist_index_no"].notna().any()
sql_shot_input = shot_sql(cfg, df_lot_hist_apc) if has_shot_ids else None
df_shot_input_apc = fetch_frame("psm_shot", sql_shot_input) if sql_shot_input else pd.DataFrame()
df_psm_input = prepare_psm_input(df_shot_input_apc, df_lot_hist_apc, df_lotinfo_adi)
save_stage("12_PSM_SHOT_QUERY", df_shot_input_apc, existing_parquet=snapshot_path("psm_shot"))
save_stage("12_PSM_INPUT", df_psm_input)


## 13. 측정점에서 기적용 PSM 값 평가
이 셀은 기존 계수를 측정점에 입히는 부분입니다. raw 계산은 다음 셀에서 따로 실행합니다. PSM 키·계수 NaN·좌표 처리 규칙은 이 함수 안에서 수정할 수 있습니다.


In [ ]:
def cpe_k_to_fit(rx, ry):
    X_dx = np.vstack([
        np.ones(len(rx)),
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12,
        (rx ** 4) / 1e19, (rx ** 3 * ry) / 1e19, (rx ** 2 * ry ** 2) / 1e19, (rx * ry ** 3) / 1e19, (ry ** 4) / 1e19,
        (rx ** 5) / 1e23, (rx ** 4 * ry) / 1e23, (rx ** 3 * ry ** 2) / 1e23, (rx ** 2 * ry ** 3) / 1e23, (rx * ry ** 4) / 1e23, (ry ** 5) / 1e23,
        (rx ** 6) / 1e27, (rx ** 5 * ry) / 1e27, (rx ** 4 * ry ** 2) / 1e27, (rx ** 3 * ry ** 3) / 1e27, (rx ** 2 * ry ** 4) / 1e27, (rx * ry ** 5) / 1e27, (ry ** 6) / 1e27,
        (rx ** 7) / 1e31, (rx ** 6 * ry) / 1e31, (rx ** 5 * ry ** 2) / 1e31, (rx ** 4 * ry ** 3) / 1e31, (rx ** 3 * ry ** 4) / 1e31, (rx ** 2 * ry ** 5) / 1e31, (rx * ry ** 6) / 1e31, (ry ** 7) / 1e31
    ]).T

    X_dy = np.vstack([
        np.ones(len(ry)),
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12, (rx ** 3) / 1e12,
        (ry ** 4) / 1e19, (ry ** 3 * rx) / 1e19, (ry ** 2 * rx ** 2) / 1e19, (ry * rx ** 3) / 1e19, (rx ** 4) / 1e19,
        (ry ** 5) / 1e23, (ry ** 4 * rx) / 1e23, (ry ** 3 * rx ** 2) / 1e23, (ry ** 2 * rx ** 3) / 1e23, (ry * rx ** 4) / 1e23, (rx ** 5) / 1e23,
        (ry ** 6) / 1e27, (ry ** 5 * rx) / 1e27, (ry ** 4 * rx ** 2) / 1e27, (ry ** 3 * rx ** 3) / 1e27, (ry ** 2 * rx ** 4) / 1e27, (ry * rx ** 5) / 1e27, (rx ** 6) / 1e27,
        (ry ** 7) / 1e31, (ry ** 6 * rx) / 1e31, (ry ** 5 * rx ** 2) / 1e31, (ry ** 4 * rx ** 3) / 1e31, (ry ** 3 * rx ** 4) / 1e31, (ry ** 2 * rx ** 5) / 1e31, (ry * rx ** 6) / 1e31, (rx ** 7) / 1e31
    ]).T

    return X_dx, X_dy

def fit_applied_psm(frame, psm_input):
    df = frame.copy()
    df["psm_fit_x"], df["psm_fit_y"], df["psm_fit_status"] = np.nan, np.nan, "missing_psm"
    if not df.empty:
        require_columns(df, ["LOT_ID", "Wafer", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y", "X_reg_demrc", "Y_reg_demrc"], "PSM 적용")
        df["_psm_lot_key"], df["_psm_slot_key"] = df["LOT_ID"].astype(str).str.strip(), integer(df["Wafer"])
        df["sub_eqp_id"] = chuck_for_slot(df["Wafer"])
        df["fcp_x_round"], df["fcp_y_round"] = pd.to_numeric(df["fcp_x"], errors="coerce").round(2), pd.to_numeric(df["fcp_y"], errors="coerce").round(2)
        df["match_key"] = psm_key(df["_psm_lot_key"], df["sub_eqp_id"], df["fcp_x_round"], df["fcp_y_round"])
        if not psm_input.empty:
            require_columns(psm_input, ["match_key"] + PSM_COLUMNS, "PSM 계수")
            source = psm_input.copy()
            sort_cols = [col for col in ["event_tsdt", "tkin_tsdt"] if col in source]
            for col in sort_cols:
                source[col] = pd.to_datetime(source[col], errors="coerce")
            source = source.sort_values(sort_cols) if sort_cols else source
            source = source.dropna(subset=["match_key"]).drop_duplicates("match_key", keep="last")
            coefficients = np.ascontiguousarray(numeric(source, PSM_COLUMNS)[PSM_COLUMNS].to_numpy(float))
            lookup = {key: i for i, key in enumerate(source["match_key"])}
            rx, ry = pd.to_numeric(df["coordinate_X"], errors="coerce").to_numpy(float), pd.to_numeric(df["coordinate_Y"], errors="coerce").to_numpy(float)
            fit_x, fit_y, status = np.full(len(df), np.nan), np.full(len(df), np.nan), df["psm_fit_status"].to_numpy(object)
            for key, positions in df.groupby("match_key", sort=False, dropna=False).indices.items():
                idx = lookup.get(key)
                if idx is None:
                    continue
                values = coefficients[idx]
                if np.isnan(values).all():
                    status[positions] = "invalid_psm_k"
                    continue
                if np.isnan(rx[positions]).any() or np.isnan(ry[positions]).any():
                    status[positions] = "invalid_psm_coordinates"
                    continue
                # 원본 010.py: 부분 NaN 계수는 0, 모든 계수가 NaN이면 해당 shot을 건너뜁니다.
                values = np.nan_to_num(values, nan=0.0)
                mx, my = cpe_k_to_fit(rx[positions], ry[positions])
                fit_x[positions], fit_y[positions] = mx.dot(values[::2]), my.dot(values[1::2])
                status[positions] = "matched"
            df["psm_fit_x"], df["psm_fit_y"], df["psm_fit_status"] = fit_x, fit_y, status
    return df

df_adi_psm_fit = fit_applied_psm(df_adi_demrc, df_psm_input)
df_oco_psm_fit = fit_applied_psm(df_oco_demrc, df_psm_input)
psm_preview = ["LOT_ID", "Wafer", "TEST", "sub_eqp_id", "fcp_x", "fcp_y", "PSM_X", "PSM_Y", "psm_fit_x", "psm_fit_y", "psm_fit_status"]
save_stage("13_ADI_PSM_FIT", df_adi_psm_fit, columns=psm_preview)
save_stage("13_OCO_PSM_FIT", df_oco_psm_fit, columns=psm_preview)


## 14. PSM decorrect → raw 계산
ADI와 OCO의 raw 계산식을 여기서 직접 확인·수정합니다. 매칭 누락으로 계산할 수 없는 값은 NaN으로 유지합니다. 이 결과에는 마지막 MMO_MRC_EQP 필터가 아직 적용되지 않았습니다.


In [ ]:
df_adi_raw = df_adi_psm_fit.copy()
df_oco_raw = df_oco_psm_fit.copy()
for frame in [df_adi_raw, df_oco_raw]:
    for col in ["PSM_X", "PSM_Y"]:
        if col not in frame:
            frame[col] = np.nan
    numeric(frame, ["PSM_X", "PSM_Y"])

# ADI: 실제 PERSHOT 보정값을 되돌림
df_adi_raw["raw_x"] = df_adi_raw["X_reg_demrc"] - df_adi_raw["PSM_X"]
df_adi_raw["raw_y"] = df_adi_raw["Y_reg_demrc"] - df_adi_raw["PSM_Y"]
df_adi_raw["psm_source"] = "PERSHOT"

# OCO: 기적용 PSM 계수로 계산한 보정값을 되돌림
df_oco_raw["raw_x"] = df_oco_raw["X_reg_demrc"] - df_oco_raw["psm_fit_x"]
df_oco_raw["raw_y"] = df_oco_raw["Y_reg_demrc"] - df_oco_raw["psm_fit_y"]
df_oco_raw["psm_source"] = "APC_PSM_FIT"
for frame in [df_adi_raw, df_oco_raw]:
    frame["decorrect_valid"] = np.isfinite(frame[["raw_x", "raw_y"]].to_numpy(float)).all(axis=1)
raw_preview = ["LOT_ID", "Wafer", "TEST", "X_reg", "Y_reg", "X_reg_demrc", "Y_reg_demrc", "PSM_X", "PSM_Y", "psm_fit_x", "psm_fit_y", "raw_x", "raw_y", "decorrect_valid"]
save_stage("14_ADI_RAW_BEFORE_FILTER", df_adi_raw, columns=raw_preview)
save_stage("14_OCO_RAW_BEFORE_FILTER", df_oco_raw, columns=raw_preview)


## 15. 원본의 마지막 필터·wafer 요약·최종 결과
MMO_MRC_EQP가 없는 행만 원본처럼 제외합니다. 보정 누락 행을 추가로 제거하지 않습니다. df_final_adi/oco, df_wafer_summary와 df_saved_files가 최종 확인 변수입니다.


In [ ]:
def summarize(frame, mode):
    if frame.empty:
        return pd.DataFrame(columns=["mode", "LOT_ID", "Wafer", "rows", "mrc_matched", "psm_fit_matched", "valid_raw", "invalid_raw"])
    work = frame[["LOT_ID", "Wafer", "decorrect_valid", "mrc_status", "psm_fit_status"]].copy()
    work["mrc_matched"], work["psm_fit_matched"] = work["mrc_status"].eq("matched"), work["psm_fit_status"].eq("matched")
    out = work.groupby(["LOT_ID", "Wafer"], dropna=False).agg(rows=("decorrect_valid", "size"), mrc_matched=("mrc_matched", "sum"), psm_fit_matched=("psm_fit_matched", "sum"), valid_raw=("decorrect_valid", "sum")).reset_index()
    out.insert(0, "mode", mode)
    out["invalid_raw"] = out["rows"] - out["valid_raw"]
    return out

df_final_adi = df_adi_raw[df_adi_raw["MMO_MRC_EQP"].notna() & df_adi_raw["MMO_MRC_EQP"].astype(str).str.strip().ne("")].copy()
df_final_oco = df_oco_raw[df_oco_raw["MMO_MRC_EQP"].notna() & df_oco_raw["MMO_MRC_EQP"].astype(str).str.strip().ne("")].copy()
df_wafer_summary = pd.concat([summarize(df_final_adi, "ADI"), summarize(df_final_oco, "OCO")], ignore_index=True)
save_stage("15_ADI_FINAL", df_final_adi, columns=raw_preview)
save_stage("15_OCO_FINAL", df_final_oco, columns=raw_preview)
save_stage("15_WAFER_SUMMARY", df_wafer_summary)
df_saved_files = pd.DataFrame(stage_files.values())
display(df_saved_files)
display(FileLink(os.path.relpath(OUTPUT_DIR / "saved_files.csv", Path.cwd()), result_html_prefix="전체 중간 파일 목록: "))
source.save_manifest(cfg)
print("MRC/PSM decorrect까지 완료했습니다.")


## 수정·재실행 메모
- 대상 선정 변경: 설정/조회/동일 wafer 셀부터 다시 실행합니다. replay 모드에서는 저장된 SQL 조건과 일치해야 합니다.
- batch 선택 변경: 06 셀의 함수를 수정하고 06 이후를 다시 실행합니다. DB 조회를 반복할 필요가 없습니다.
- 좌표/메타데이터 결합 변경: 08부터 다시 실행합니다.
- MRC 계산 변경: 10의 행렬/함수를 수정하고 10, 13, 14, 15를 다시 실행합니다.
- PSM 평가 변경: 13부터 다시 실행합니다. raw 계산식만 변경하면 14, 15만 다시 실행합니다.
- 저장 파일로 복원: `df_adi_demrc = pd.read_parquet(STAGE_DIR / "10_ADI_DEMRC.parquet")`처럼 필요한 변수를 읽습니다. 복원해서 계산할 함수와 계수 변수도 먼저 준비해야 합니다.
- 중간 테이블에는 원래 측정값이 보존됩니다. 메모리가 부족하면 확인이 끝난 대형 조회 변수만 직접 삭제하고 저장 파일에서 다시 읽을 수 있습니다.

검증은 합성 입력에서 원본 및 모듈 버전과 비교합니다. 생산 데이터와 사내 DB 결과는 사내 환경에서 실행해 확인해야 합니다. 이 파일은 GitHub에 자동으로 게시되지 않습니다.
